# Sem 4: Image to Voxel Reconstruction

Готовый ноутбук для задания по восстановлению 3D voxel-представления по одному изображению детали.
- Данные готовы: `dataset_summary.json` и voxel-кеш в `artifacts/voxel_cache_32`.
- Attention реализован: `SpatialSelfAttention2d` в `sem4_image_to_voxel_pipeline.py`.
- Первая модель обучена: `model_1_baseline.pt`.
- Вторая модель обучена: `model_2_attention.pt`.
- Третья модель обучена: `model_3_residual_attention.pt`.


## Связь с исходными ноутбуками

- `Image_to_Voxel.ipynb` использован как основа задачи: PNG -> STL -> voxel target -> image-to-voxel модель.
- `Video_to_3D_SfM_to_Neural.ipynb` изучен как референс 3D reconstruction/SfM-пайплайна. В текущем датасете уже есть пары `png + stl`, поэтому обучение идет напрямую по этим парам.
- Исходный `Image_to_Voxel.ipynb` ожидал папку `data_3dmodel` и `trimesh`; в этом решении используется фактическая папка `dataset` и встроенный бинарный STL-парсер без зависимости от `trimesh`.


In [1]:
from pathlib import Path
import json

ROOT = Path.cwd()
if ROOT.name != 'sem_4':
    ROOT = ROOT / 'sem_4'

print('Корень sem_4:', ROOT.resolve())
print('Датасет существует:', (ROOT / 'dataset').exists())

Корень sem_4: C:\python\try_to_nn\CV_ITMO\sem_4
Датасет существует: True


## 1. Подготовка данных

Пайплайн находит файлы `*.png` и `*.stl` с одинаковым stem, читает бинарную STL-геометрию, нормализует вершины и растеризует поверхность в voxel-grid `32 x 32 x 32`. Кеш сохраняется в `artifacts/voxel_cache_32`.


In [2]:
# Recreate or validate voxel cache if needed.
import sys
!{sys.executable} sem4_image_to_voxel_pipeline.py --mode inspect --max-samples 96


{
  "dataset_dir": "C:\\python\\try_to_nn\\CV_ITMO\\sem_4\\dataset",
  "num_pairs": 561,
  "cached_pairs": 96,
  "voxel_size": 32,
  "image_size": 64,
  "mean_occupancy": 0.1556987762451172,
  "first_png": "C:\\python\\try_to_nn\\CV_ITMO\\sem_4\\dataset\\part_0000.png",
  "first_stl": "C:\\python\\try_to_nn\\CV_ITMO\\sem_4\\dataset\\part_0000.stl",
  "first_image_size": [
    1024,
    1024
  ],
  "first_image_mode": "RGBA",
  "first_voxel_shape": [
    32,
    32,
    32
  ],
  "first_voxel_occupancy": 0.14410400390625
}


## 2. Attention

Attention реализован в `SpatialSelfAttention2d`. Карта признаков `B x C x H x W` преобразуется в токены, проходит через `nn.MultiheadAttention`, затем через feed-forward блок и возвращается в spatial-форму. Блок используется во второй и третьей моделях.


In [3]:
from sem4_image_to_voxel_pipeline import SpatialSelfAttention2d, make_model

attn = SpatialSelfAttention2d(channels=256, heads=4)
print(attn)

model_1 = make_model('baseline')
model_2 = make_model('attention')
model_3 = make_model('residual_attention')
print(type(model_1).__name__, type(model_2).__name__, type(model_3).__name__)

SpatialSelfAttention2d(
  (norm): GroupNorm(1, 256, eps=1e-05, affine=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (ffn): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=512, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=512, out_features=256, bias=True)
  )
)
ImageToVoxelNet ImageToVoxelNet ImageToVoxelNet


## 3. Обучение трех моделей

Короткий контрольный прогон уже выполнен на 96 примерах, 2 эпохи, batch size 8. Для более высокого качества увеличь `--epochs` и `--max-samples`.


In [4]:
# Full GPU training is intentionally launched from PowerShell to avoid accidental overwrites.
print(r'C:\anacon\envs\cv_env\python.exe sem_4\sem4_image_to_voxel_pipeline.py --mode train --device cuda --require-gpu --amp --epochs 10 --batch-size 8 --max-samples 561 --num-workers 0')


C:\anacon\envs\cv_env\python.exe sem_4\sem4_image_to_voxel_pipeline.py --mode train --device cuda --require-gpu --amp --epochs 10 --batch-size 8 --max-samples 561 --num-workers 0


## 4. Результаты обучения

Результаты сохранены в `artifacts/training_results.csv` и `artifacts/training_results.json`. Чекпоинты находятся в `artifacts/checkpoints`.


In [5]:
import csv

results_path = ROOT / 'artifacts' / 'training_results.csv'
with results_path.open('r', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
for row in rows:
    print(row['name'], 'attention=', row['use_attention'], 'best_epoch=', row['best_epoch'], 'val_iou=', row['val_iou'], 'val_dice=', row['val_dice'])

model_1_baseline attention= False best_epoch= 10 val_iou= 0.26243640879789987 val_dice= 0.4041428744792938
model_2_attention attention= True best_epoch= 11 val_iou= 0.3105873097976049 val_dice= 0.45809855063756305
model_3_residual_attention attention= True best_epoch= 12 val_iou= 0.3132423837979635 val_dice= 0.4612146397431692


## 5. Чекпоинты

Ячейка ниже проверяет, что все три чекпоинта обученных моделей существуют.


In [6]:
from pathlib import Path

checkpoint_dir = ROOT / 'artifacts' / 'checkpoints'
for name in ['model_1_baseline.pt', 'model_2_attention.pt', 'model_3_residual_attention.pt']:
    path = checkpoint_dir / name
    print(name, path.exists(), path.stat().st_size if path.exists() else 0)

model_1_baseline.pt True 29438871
model_2_attention.pt True 31551590
model_3_residual_attention.pt True 31552357


## 6. Команда для более долгого обучения

Обязательные чекпоинты уже созданы. Для улучшения качества можно запустить более длинное обучение:

```powershell
C:\anacon\envs\cv_env\python.exe sem_4\sem4_image_to_voxel_pipeline.py --mode train --epochs 10 --batch-size 8 --max-samples 300
```

Для полного датасета можно указать `--max-samples 561`.
